In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="VjAxZSEJru3p0Pij8zYu")
project = rf.workspace("mehdis-workspace-jupje").project("smart-football-object-detection-apwnj")
version = project.version(1)
dataset = version.download("yolov8")


Defaulting to user installation because normal site-packages is not writeable
loading Roboflow workspace...
loading Roboflow project...


NotADirectoryError: [WinError 267] Nom de répertoire non valide: 'Smart-Football:-Object-Detection-1'

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import yaml

# Lire le fichier généré par Roboflow
with open(f"{dataset.location}/data.yaml", 'r') as stream:
    data = yaml.safe_load(stream)
    print(data)
# print("Nombre de classes (nc) :", data['nc'])
# print("Noms des classes :", data['names'])

{'names': ['Ball', 'Keeper', 'Player', 'Ref'], 'nc': 4, 'roboflow': {'license': 'CC BY 4.0', 'project': 'smart-football-object-detection-apwnj', 'url': 'https://universe.roboflow.com/mehdis-workspace-jupje/smart-football-object-detection-apwnj/dataset/1', 'version': 1, 'workspace': 'mehdis-workspace-jupje'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


In [14]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.2 MB/s eta 0:00:00


In [1]:
# Exemple de ce que vous feriez juste après :
from ultralytics import YOLO

# model = YOLO("/content/drive/MyDrive/best.pt")  # Charge un modèle YOLOv8 natif
model = YOLO("../fine-tunning_1/runs/detect/train/weights/best.pt")  # Charge un modèle YOLOv8 natif
# model.train(data=f"{dataset.location}/data.yaml", epochs=100)  # Lance l'entraînement !

In [2]:
resultat  = model("../../../data/extracted_frames/data1/frame_1_490.jpg")
resultat[0].show()


image 1/1 D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\detection\fine-tunning2\..\..\..\data\extracted_frames\data1\frame_1_490.jpg: 384x640 3 Balls, 24 Players, 2 Refs, 76.9ms
Speed: 3.9ms preprocess, 76.9ms inference, 4.4ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
# 2. Lancer la validation exclusivement sur le jeu de test
metrics = model.val(split='test')

# 3. Afficher les résultats clés
print(f"mAP50-95 (Précision globale) : {metrics.box.map:.4f}")
print(f"mAP50 (Précision à seuil 0.5) : {metrics.box.map50:.4f}")
print(f"Précision par classe : {metrics.box.mp:.4f}")
print(f"Rappel (Recall) par classe : {metrics.box.mr:.4f}")

# Afficher les performances détaillées par classe
for i, name in enumerate(metrics.names.values()):
    print(f"--- Classe : {name} ---")
    print(f"  Précision (Precision) : {metrics.box.class_result(i)[0]:.4f}")
    print(f"  Rappel (Recall)       : {metrics.box.class_result(i)[1]:.4f}")
    print(f"  mAP50                 : {metrics.box.class_result(i)[2]:.4f}")
    print("-" * 25)

Ultralytics 8.4.60  Python-3.13.2 torch-2.12.0+cpu CPU (Intel Core i5-1035G1 1.00GHz)
val: Fast image access  (ping: 4.69.4 ms, read: 1.10.4 MB/s, size: 46.1 KB)
val: Scanning D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\match-1\test\labels... 14 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14/14 64.1it/s 0.2s
val: New cache created: D:\MEHDI\Study\WISD\S2\Visual analytics\projet\Analyse-Tactique-du-Football-par-Vision-par-Ordinateur\notebooks\match-1\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.9s/it 2.9s
                   all         14        218      0.885      0.781      0.842      0.425
                  Ball         14         14      0.852      0.415      0.476      0.201
                Keeper          9          9      0.805      0.778      0.915      0.407
                Player         14        169      0.948     

In [4]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs\detect\train\weights\best.pt")
cap = cv2.VideoCapture(r"..\video1.mp4")

# Récupérer les propriétés de la vidéo d'origine
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Créer le fichier de sortie
out = cv2.VideoWriter(r".\resultat_annote1.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    resultats = model(frame,conf=0.3 , iou = 0.5)
    frame_annotee = resultats[0].plot()

    out.write(frame_annotee)        # écrit la frame dans le fichier vidéo
    cv2.imshow("Detection", frame_annotee)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


0: 384x640 1 Keeper, 16 Players, 1 Ref, 57.1ms
Speed: 2.4ms preprocess, 57.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 19 Players, 2 Refs, 75.7ms
Speed: 3.7ms preprocess, 75.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 19 Players, 65.3ms
Speed: 2.0ms preprocess, 65.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 20 Players, 63.9ms
Speed: 1.9ms preprocess, 63.9ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 19 Players, 71.2ms
Speed: 3.0ms preprocess, 71.2ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 20 Players, 1 Ref, 61.2ms
Speed: 2.0ms preprocess, 61.2ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Keeper, 21 Players, 60.7ms
Speed: 3.7ms preprocess, 60.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 